In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import transformers
import datasets
import torch
import pandas as pd
from tqdm import tqdm
import pickle
from transformer_lens import HookedTransformer, utils
import einops
import pickle
import os
from datetime import datetime
import lm_eval

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

left_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
left_tokenizer.pad_token = left_tokenizer.eos_token
left_tokenizer.padding_side = "left"

right_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
right_tokenizer.pad_token = right_tokenizer.eos_token

lat_models = {}
# lat_models["WHP"] = AutoModelForCausalLM.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter", torch_dtype=torch.bfloat16)
# lat_models["LLaMA"] = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
base_models = {"WHP": "microsoft/Llama2-7b-WhoIsHarryPotter", "LLaMA": "meta-llama/Llama-2-7b-chat-hf"}

lat_model_names = {
    "PCA_L8_Eps0.1": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=0.1-pgd_layer=82024-04-24-04-44-24",
    "PCA_L8_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-39-29",
    "PCA_L8_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-36-59",
    "PCA_L15_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=1.0-pgd_layer=152024-04-24-06-56-25",
    "PCA_L15_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=10.0-pgd_layer=152024-04-24-06-57-07",
    "No_PCA_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25",
    "No_PCA_L8_Eps10": "models/hp-lat-llama-None-2024-04-10-16-09-25",
    "No_PCA_L15_Eps1": "models/hp-lat-llama-None-epsilon=1.0-pgd_layer=152024-04-24-06-54-54",
    "No_PCA_L15_Eps10": "models/hp-lat-llama-None-epsilon=10.0-pgd_layer=152024-04-24-06-55-04",
    "WHP_Replication": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=02024-04-24-03-28-01",
    "WHP_All_Coefs": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=82024-04-18-18-42-03",
}

merge_and_unload = False
# lat_model_names = {"WHP_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25", "SAQ_L8_Eps1": "models/hp-lat-llama-None-2024-04-03-09-29-58"}
# for short_name, model_name in lat_model_names.items():
#     lat_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
#     lat_model = PeftModel.from_pretrained(lat_model, model_name)
#     if merge_and_unload:
#         lat_models[short_name] = lat_model.merge_and_unload()
#     else:
#         lat_models[short_name] = lat_model

save_dir = f"results/llama-lat-less-noise-2"
os.makedirs(save_dir, exist_ok=True)

In [6]:
from lm_eval import evaluate
from lm_eval.models.huggingface import HFLM

# for model_name in 
# model = lat_models["LLaMA"]
# model.cpu()
model = HFLM(pretrained="meta-llama/Llama-2-7b-chat-hf", dtype=torch.bfloat16, device="cuda")

# model = HFLM(pretrained="meta-llama/Llama-2-7b-chat-hf", peft=lat_model_names)

# model.cuda()
# task_dict = lm_eval.tasks.get_task_dict(
#     ["mmlu", "sciq"]
# )

results = lm_eval.simple_evaluate(
    model=model,
    tasks=["mmlu", "sciq"]
)

2024-05-02:12:08:22,593 WARNING  [logging.py:61] Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2024-05-02:12:08:22,594 INFO     [huggingface.py:164] Using device 'cuda'


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2024-05-02:12:08:30,799 INFO     [evaluator.py:131] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2024-05-02:12:08:30,800 INFO     [evaluator.py:191] Using pre-initialized model
/data/phillip_guo/miniconda3/envs/hp-unlrn/lib/python3.10/site-packages/datasets/load.py:1429: FutureWarning: The repository for hails/mmlu_no_train contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/hails/mmlu_no_train
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

...

Generating dev split: 0 examples [00:00, ? examples/s]

Generating train split:   0%|          | 0/11679 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

2024-05-02:12:10:37,649 INFO     [task.py:395] Building contexts for sciq on rank 0...
100%|██████████| 1000/1000 [00:00<00:00, 1018.75it/s]
2024-05-02:12:10:38,671 INFO     [task.py:395] Building contexts for mmlu_world_religions on rank 0...
100%|██████████| 171/171 [00:00<00:00, 814.49it/s]
2024-05-02:12:10:38,889 INFO     [task.py:395] Building contexts for mmlu_prehistory on rank 0...
100%|██████████| 324/324 [00:00<00:00, 831.64it/s]
2024-05-02:12:10:39,291 INFO     [task.py:395] Building contexts for mmlu_high_school_european_history on rank 0...
100%|██████████| 165/165 [00:00<00:00, 831.12it/s]
2024-05-02:12:10:39,498 INFO     [task.py:395] Building contexts for mmlu_professional_law on rank 0...
100%|██████████| 1534/1534 [00:01<00:00, 826.33it/s]
2024-05-02:12:10:41,410 INFO     [task.py:395] Building contexts for mmlu_high_school_world_history on rank 0...
100%|██████████| 237/237 [00:00<00:00, 821.43it/s]
2024-05-02:12:10:41,710 INFO     [task.py:395] Building contexts for

In [8]:
model.cpu()

AttributeError: 'HFLM' object has no attribute 'cpu'

In [11]:
model = HFLM(pretrained="meta-llama/Llama-2-7b-chat-hf", peft=lat_model_names["WHP_Replication"], dtype=torch.bfloat16, device="cuda")

2024-05-02:12:56:18,356 WARNING  [logging.py:61] Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2024-05-02:12:56:18,357 INFO     [huggingface.py:164] Using device 'cuda'


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [15]:
del model

In [16]:
torch.cuda.memory_allocated() // 1024**3

25

In [18]:
torch.cuda.empty_cache()
torch.cuda.memory_allocated() // 1024**3

12

In [ ]:
model = HFLM(pretrained="meta-llama/Llama-2-7b-chat-hf", dtype=torch.bfloat16, device="cuda")

results = lm_eval.simple_evaluate(
    model=model,
    tasks=["mmlu", "sciq"]
)